# 01 - Build Panel

Merges OECD STAN, BERD, GBAORD, GDP, and R&D tax credit data into a country × industry × year panel (1987–2009).
Constructs log variables, lags, and employment-share weights for WLS estimation.

## Loading raw files

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

RAW = Path("../data/raw")

# Load dimension lookup tables
dim_ind = pd.read_csv(RAW / "dimIND.txt", sep="|", encoding="latin-1")
dim_cou = pd.read_csv(RAW / "dimCOU.txt", sep="|", encoding="latin-1", header=None, names=["cocode", "country_en", "country_fr", "id"])
dim_var = pd.read_csv(RAW / "dimVAR.txt", sep="|", encoding="latin-1")

In [2]:
# Load and filter STAN to needed variables
stan_raw = pd.read_csv(RAW / "DATA.txt", sep="|")

keep_vars = ["VALU", "EMPN", "CPGK"]
stan = stan_raw[stan_raw["Var"].isin(keep_vars)].copy()

# Pivot wide: one row per country-industry-year
stan_wide = stan.pivot_table(
    index=["Cou", "ind", "year"],
    columns="Var",
    values="value"
).reset_index()

stan_wide.columns.name = None
stan_wide.rename(columns={
    "Cou": "country",
    "ind": "ind4digit"
}, inplace=True)

print(stan_wide.shape)
print(stan_wide.head())
print(stan_wide["ind4digit"].nunique(), "industries")
print(stan_wide["country"].nunique(), "countries")

(84606, 6)
  country ind4digit  year  CPGK      EMPN  VALU
0     AUS      0102  1980   NaN  399475.0   NaN
1     AUS      0102  1981   NaN  395002.0   NaN
2     AUS      0102  1982   NaN  402475.0   NaN
3     AUS      0102  1983   NaN  394559.0   NaN
4     AUS      0102  1984   NaN  392572.0   NaN
107 industries
34 countries


In [3]:
# Load BERD expenditure by source of funds and industry
berd_funds_raw = pd.read_csv(RAW / "BERD_FUNDS_Data_bf1eacf7-45e8-42af-b865-f56b3f5dd72d.csv")
print(berd_funds_raw.shape)
print(berd_funds_raw.head())
print(berd_funds_raw.columns.tolist())
print(berd_funds_raw["FUND"].unique() if "FUND" in berd_funds_raw.columns else berd_funds_raw.iloc[:,2].unique())

(85723, 7)
                  Units for Expenditure    Country      Source of Funds  \
0  Million PPP Dollars - Current prices  Australia  Business enterprise   
1  Million PPP Dollars - Current prices  Australia  Business enterprise   
2  Million PPP Dollars - Current prices  Australia  Business enterprise   
3  Million PPP Dollars - Current prices  Australia  Business enterprise   
4  Million PPP Dollars - Current prices    Austria  Business enterprise   

                            Industry  Year   Value Flags  
0  AGRICULTURE, HUNTING AND FORESTRY  2005  62.232     s  
1  AGRICULTURE, HUNTING AND FORESTRY  2006  79.180   NaN  
2  AGRICULTURE, HUNTING AND FORESTRY  2007  77.163   NaN  
3  AGRICULTURE, HUNTING AND FORESTRY  2008  94.316   NaN  
4  AGRICULTURE, HUNTING AND FORESTRY  1989   0.395   NaN  
['Units for Expenditure', 'Country', 'Source of Funds', 'Industry', 'Year', 'Value', 'Flags']
<StringArray>
[   'Business enterprise',   'Sub-total government',         'Other National

## Cleaning GBAORD, GDP, and tax credits

In [4]:
# Load government budget appropriations for R&D (GBAORD)
gbaord_raw = pd.read_csv(RAW / "GBAORD_NABS2007_Data_00b5d1e4-b62b-42ae-b54c-7465e7bcdd7a.csv")
print(gbaord_raw.shape)
print(gbaord_raw.head())
print(gbaord_raw.columns.tolist())
print(gbaord_raw.iloc[:, 2].unique())  # likely the category column

(28123, 6)
                            Units for Expenditure  \
0  Million National Currency (Euro For Euro Area)   
1  Million National Currency (Euro For Euro Area)   
2  Million National Currency (Euro For Euro Area)   
3  Million National Currency (Euro For Euro Area)   
4  Million National Currency (Euro For Euro Area)   

             GBAORD Socio economic objective    Country  Year  Value Flags  
0  Exploration and exploitation of the Earth  Australia  1981   35.4     a  
1  Exploration and exploitation of the Earth  Australia  1982   45.7     c  
2  Exploration and exploitation of the Earth  Australia  1983   89.1    ah  
3  Exploration and exploitation of the Earth  Australia  1984   88.3     h  
4  Exploration and exploitation of the Earth  Australia  1985   90.2    ah  
['Units for Expenditure', 'GBAORD Socio economic objective', 'Country', 'Year', 'Value', 'Flags']
<StringArray>
[         'Australia',            'Austria',            'Belgium',
             'Canada',     'C

In [5]:
# Filter to defence category
gbaord = gbaord_raw[
    gbaord_raw["GBAORD Socio economic objective"] == "Defence"
].copy()

# Keep relevant columns
gbaord = gbaord[["Country", "Year", "Value"]].copy()
gbaord.rename(columns={
    "Country": "country",
    "Year": "year",
    "Value": "gbaord_defence"
}, inplace=True)

print(gbaord.shape)
print(gbaord.head())
print(gbaord["country"].nunique(), "countries")
print(gbaord["year"].min(), "to", gbaord["year"].max())

(2187, 3)
       country  year  gbaord_defence
957  Australia  1981           112.9
958  Australia  1982           141.4
959  Australia  1983           121.2
960  Australia  1984           133.0
961  Australia  1985           139.1
35 countries
1981 to 2010


In [6]:
# Load OECD national accounts (SNA Table 1) for GDP
sna1_raw = pd.read_csv(RAW / "SNA_TABLE1_Data_0ceb103f-b307-4940-995c-b751c31b8835.csv")
print(sna1_raw.shape)
print(sna1_raw.columns.tolist())
print(sna1_raw.iloc[:, 2].unique())  # transaction types
print(sna1_raw.iloc[:, 3].unique())  # measure types

(37755, 7)
['Transaction', 'Measure', 'Frequency', 'Country', 'Time', 'Value', 'Flags']
<StringArray>
['Annual']
Length: 1, dtype: str
<StringArray>
[                         'Australia',                            'Austria',
                            'Belgium',                             'Canada',
                     'Czech Republic',                            'Denmark',
                            'Finland',                             'France',
                            'Germany',                             'Greece',
                            'Hungary',                            'Iceland',
                            'Ireland',                              'Italy',
                              'Japan',                              'Korea',
                         'Luxembourg',                             'Mexico',
                        'Netherlands',                        'New Zealand',
                             'Norway',                             'Poland',
    

In [7]:
gdp = sna1_raw[
    (sna1_raw["Transaction"] == "Gross domestic product (expenditure approach)") &
    (sna1_raw["Measure"] == "US $, current prices, current PPPs, millions")
].copy()

# Drop aggregate regions
drop_countries = [
    "OECD - Europe", "OECD - Total", "OECD - FORMER TOTAL",
    "European Union (15 countries)", "European Union (27 countries)",
    "Euro area (17 countries)"
]
gdp = gdp[~gdp["Country"].isin(drop_countries)].copy()

# Clean columns
gdp = gdp[["Country", "Time", "Value"]].copy()
gdp.rename(columns={
    "Country": "country",
    "Time": "year",
    "Value": "gdp"
}, inplace=True)

print(gdp.shape)
print(gdp.head())
print(gdp["country"].nunique(), "countries")
print(gdp["year"].min(), "to", gdp["year"].max())

(1502, 3)
       country  year           gdp
426  Australia  1960  25081.819396
427  Australia  1961  25366.306650
428  Australia  1962  27958.264513
429  Australia  1963  30435.843426
430  Australia  1964  32746.700948
41 countries
1960 to 2013


In [8]:
# Load R&D tax credit B-index data
rdtax_raw = pd.read_csv(RAW / "rdtaxcredits.csv")
print(rdtax_raw.shape)
print(rdtax_raw.head())
print(rdtax_raw.columns.tolist())

(702, 8)
     Country  year   CIT labour other current    M&E    B&E foreign
0  Australia  1980  0.46      1             1  1.075  1.075   1.008
1  Australia  1981  0.46      1             1  1.075  1.075   1.008
2  Australia  1982  0.46      1             1  1.075  1.075   1.008
3  Australia  1983  0.46      1             1  1.075  1.075   1.008
4  Australia  1984  0.46      1             1  1.075  1.075   1.008
['Country', 'year', 'CIT', 'labour', 'other current', 'M&E', 'B&E', 'foreign']


In [9]:
rdtax = rdtax_raw.copy()

# Convert columns to numeric (strip any dashes)
for col in ["labour", "other current", "M&E", "B&E"]:
    rdtax[col] = pd.to_numeric(
        rdtax[col].astype(str).str.replace("-", "", regex=False),
        errors="coerce"
    )

# Construct B-index and tax subsidy rate
rdtax["b"] = (0.6  * rdtax["labour"] +
              0.3  * rdtax["other current"] +
              0.05 * rdtax["M&E"] +
              0.05 * rdtax["B&E"])

rdtax["taxsubs"] = 1 - rdtax["b"]

# Fix country name formatting
rdtax["Country"] = rdtax["Country"].replace({
    "CzechRepublic":  "Czech Republic",
    "UnitedStates":   "United States",
    "UnitedKingdom":  "United Kingdom",
    "NewZealand":     "New Zealand"
})

rdtax = rdtax[["Country", "year", "taxsubs"]].copy()
rdtax.rename(columns={"Country": "country"}, inplace=True)

print(rdtax.shape)
print(rdtax.head())
print(rdtax["taxsubs"].describe())

(702, 3)
     country  year  taxsubs
0  Australia  1980  -0.0075
1  Australia  1981  -0.0075
2  Australia  1982  -0.0075
3  Australia  1983  -0.0075
4  Australia  1984  -0.0075
count    669.000000
mean       0.036533
std        0.098674
min       -0.069600
25%       -0.019950
50%       -0.009550
75%        0.077750
max        0.433450
Name: taxsubs, dtype: float64


## Building the crosswalk

In [10]:
# Load industry crosswalk: 4-digit STAN codes to BERD industry groups
import pyreadstat

ind_corr, meta = pyreadstat.read_dta(RAW / "indcorrespondence.dta")
print(ind_corr.shape)
print(ind_corr.head(20))
print(ind_corr.columns.tolist())

(94, 2)
   ind4digit                                        matchingind
0       0199                                                   
1       0105                  AGRICULTURE, HUNTING AND FORESTRY
2       0102                                                   
3       0100                                                   
4       0200                                                   
5       0500                                                   
6       1014                               MINING AND QUARRYING
7       1012                                                   
8       1314                                                   
9       1537                                                   
10      1516                        Food, beverages and tobacco
11      1500                                                   
12      1600                                                   
13      1719                          Textiles, fur and leather
14      1718                    

In [11]:
# Forward-fill the industry group name
ind_corr["matchingind"] = ind_corr["matchingind"].replace("", pd.NA).ffill()

# Check how many unique industry groups we get
print(ind_corr["matchingind"].nunique(), "industry groups")
print(ind_corr["matchingind"].unique())

26 industry groups
<StringArray>
[                                                                           nan,
                                            'AGRICULTURE, HUNTING AND FORESTRY',
                                                         'MINING AND QUARRYING',
                                                  'Food, beverages and tobacco',
                                                    'Textiles, fur and leather',
                                                'Wood and cork (not furniture)',
                         'Pulp, paper, paper products, printing and publishing',
                            'Coke, refined petroleum products and nuclear fuel',
                                              'Chemicals and chemical products',
                                                  'Rubber and plastic products',
                                                'Non-metallic mineral products',
                                                                 'Basic meta

## Cleaning and reshaping BERD

In [12]:
# Pivot wide on source of funds
berd = berd_funds_raw[
    berd_funds_raw["Units for Expenditure"] == "Million PPP Dollars - Current prices"
].copy()

berd = berd[["Country", "Source of Funds", "Industry", "Year", "Value"]].copy()
berd.rename(columns={
    "Country": "country", "Source of Funds": "source",
    "Industry": "industry", "Year": "year", "Value": "value"
}, inplace=True)

# Keep only gov and total
berd = berd[berd["source"].isin(["Sub-total government", "Total (funding sector)"])].copy()
berd["source"] = berd["source"].map({
    "Sub-total government": "gov",
    "Total (funding sector)": "tot"
})

berd_wide = berd.pivot_table(
    index=["country", "industry", "year"],
    columns="source", values="value"
).reset_index()
berd_wide.columns.name = None

# Apply the 5 indmatch replacements (exactly as in Stata)
berd_wide["indmatch"] = berd_wide["industry"]
berd_wide.loc[berd_wide["industry"].isin([
    "Railway and tramway locomotives and rolling stock",
    "Transport equipment, nec"
]), "indmatch"] = "Railroad equipment and transport equipment n.e.c."

berd_wide.loc[berd_wide["industry"].isin([
    "Furniture, other manufacturing nec", "Recycling"
]), "indmatch"] = "Manufacturing n.e.c. and recycling"

berd_wide.loc[berd_wide["industry"].isin([
    "Pulp, paper and paper products",
    "Publishing, printing and reproduction of recorded media"
]), "indmatch"] = "Pulp, paper, paper products, printing and publishing"

berd_wide.loc[berd_wide["industry"].isin([
    "Wholesale, retail trade and motor vehicle repair", "Hotels and restaurants"
]), "indmatch"] = "WHOLESALE AND RETAIL TRADE - RESTAURANTS AND HOTELS"

berd_wide.loc[berd_wide["industry"].isin([
    "Financial intermediation (includes insurance)",
    "Real estate, renting and business activities"
]), "indmatch"] = "FINANCE, INSURANCE, REAL ESTATE AND BUSINESS SERVICES"

# Track missing before summing (if any component missing, group = missing)
berd_wide["missing_gov"] = berd_wide["gov"].isna().astype(int)
berd_wide["missing_tot"] = berd_wide["tot"].isna().astype(int)

# Collapse (sum) by country x indmatch x year
berd_agg = berd_wide.groupby(["country", "indmatch", "year"]).agg(
    gov=("gov", "sum"),
    tot=("tot", "sum"),
    missing_gov=("missing_gov", "sum"),
    missing_tot=("missing_tot", "sum")
).reset_index()

# Set to NaN if any component was missing (matching Stata logic)
berd_agg.loc[berd_agg["missing_gov"] > 0, "gov"] = float("nan")
berd_agg.loc[berd_agg["missing_tot"] > 0, "tot"] = float("nan")
berd_agg.drop(columns=["missing_gov", "missing_tot"], inplace=True)

# Construct outcome variable
berd_agg["rdexgov"] = berd_agg["tot"] - berd_agg["gov"]

print(berd_agg.shape)
print(berd_agg.head())
print(berd_agg["indmatch"].nunique(), "industry groups")

(24338, 6)
     country                                           indmatch  year  gov  \
0  Argentina                                       Basic metals  1996  NaN   
1  Argentina  Coke, petroleum, nuclear fuel, chemicals and p...  1996  NaN   
2  Argentina  Fabricated metal products, except machinery an...  1996  NaN   
3  Argentina  Fabricated metal products, machinery and equip...  1996  NaN   
4  Argentina                        Food products and beverages  1996  NaN   

      tot  rdexgov  
0   4.871      NaN  
1  48.713      NaN  
2   2.012      NaN  
3  54.220      NaN  
4  55.173      NaN  
73 industry groups


## Merging into panel

In [13]:
# Merge STAN with crosswalk
stan_merged = stan_wide.merge(
    ind_corr[["ind4digit", "matchingind"]],
    on="ind4digit",
    how="inner"
)

# Drop NaN matchingind (aggregate total codes like 0199)
stan_merged = stan_merged.dropna(subset=["matchingind"])

# Aggregate to country x industry group x year by summing
stan_agg = stan_merged.groupby(
    ["country", "matchingind", "year"]
)[["VALU", "EMPN", "CPGK"]].sum(min_count=1).reset_index()

stan_agg.rename(columns={
    "matchingind": "indmatch",
    "VALU": "valu",
    "EMPN": "empn",
    "CPGK": "cpgk"
}, inplace=True)

print(stan_agg.shape)
print(stan_agg.head())
print(stan_agg["indmatch"].nunique(), "industry groups")
print(stan_agg["country"].nunique(), "countries")

(25739, 6)
  country                           indmatch  year          valu      empn  \
0     AUS  AGRICULTURE, HUNTING AND FORESTRY  1970  2.451554e+09       NaN   
1     AUS  AGRICULTURE, HUNTING AND FORESTRY  1971  2.779557e+09  412112.0   
2     AUS  AGRICULTURE, HUNTING AND FORESTRY  1972  3.780881e+09  443120.0   
3     AUS  AGRICULTURE, HUNTING AND FORESTRY  1973  5.146750e+09  427044.0   
4     AUS  AGRICULTURE, HUNTING AND FORESTRY  1974  4.522092e+09  405043.0   

   cpgk  
0   NaN  
1   NaN  
2   NaN  
3   NaN  
4   NaN  
26 industry groups
34 countries


In [14]:
# Build ISO → full name mapping
country_map = dict(zip(dim_cou["cocode"], dim_cou["country_en"]))

# Apply to STAN
stan_agg["country"] = stan_agg["country"].map(country_map)

# Check for any unmapped codes
print(stan_agg["country"].isna().sum(), "unmapped country codes")
print(stan_agg["country"].nunique(), "countries after mapping")
print(sorted(stan_agg["country"].dropna().unique()))

0 unmapped country codes
34 countries after mapping
['Australia', 'Austria', 'Belgium', 'Canada', 'Chile', 'Czech Republic', 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Israel', 'Italy', 'Japan', 'Korea', 'Luxembourg', 'Mexico', 'Netherlands', 'New Zealand', 'Norway', 'Poland', 'Portugal', 'Slovak Republic', 'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'United Kingdom', 'United States', 'West Germany']


In [15]:
# Start with STAN as backbone
panel = stan_agg.copy()

# Merge BERD on country x indmatch x year
panel = panel.merge(berd_agg, on=["country", "indmatch", "year"], how="left")

# Merge GBAORD on country x year
panel = panel.merge(gbaord, on=["country", "year"], how="left")

# Merge GDP on country x year
panel = panel.merge(gdp, on=["country", "year"], how="left")

# Merge tax subsidies on country x year
panel = panel.merge(rdtax, on=["country", "year"], how="left")

# Drop West Germany
panel = panel[panel["country"] != "West Germany"].copy()

print(panel.shape)
print(panel.head())
print(panel.isnull().mean().round(3))

(57737, 12)
     country                           indmatch  year          valu      empn  \
0  Australia  AGRICULTURE, HUNTING AND FORESTRY  1970  2.451554e+09       NaN   
1  Australia  AGRICULTURE, HUNTING AND FORESTRY  1971  2.779557e+09  412112.0   
2  Australia  AGRICULTURE, HUNTING AND FORESTRY  1972  3.780881e+09  443120.0   
3  Australia  AGRICULTURE, HUNTING AND FORESTRY  1973  5.146750e+09  427044.0   
4  Australia  AGRICULTURE, HUNTING AND FORESTRY  1974  4.522092e+09  405043.0   

   cpgk  gov  tot  rdexgov  gbaord_defence           gdp  taxsubs  
0   NaN  NaN  NaN      NaN             NaN  58800.750408      NaN  
1   NaN  NaN  NaN      NaN             NaN  64049.179002      NaN  
2   NaN  NaN  NaN      NaN             NaN  69753.694928      NaN  
3   NaN  NaN  NaN      NaN             NaN  78801.594222      NaN  
4   NaN  NaN  NaN      NaN             NaN  86229.224448      NaN  
country           0.000
indmatch          0.000
year              0.000
valu              0.0

## Constructing variables and lags

In [16]:
# Sample restrictions
panel = panel[panel["country"] != "West Germany"].copy()
panel = panel[panel["country"] != "Slovenia"].copy()
panel = panel[(panel["year"] >= 1987) & (panel["year"] <= 2009)].copy()

# Construct log variables (drop non-positive values first)

panel = panel[panel["rdexgov"] > 0].copy()
panel = panel[panel["gov"] > 0].copy()
panel = panel[panel["valu"] > 0].copy()
panel = panel[panel["gdp"] > 0].copy()

panel["lnrdexgov"]   = np.log(panel["rdexgov"])
panel["lnrdgov"]     = np.log(panel["gov"])
panel["lnvalu"]      = np.log(panel["valu"])
panel["lngdp"]       = np.log(panel["gdp"])

print(panel.shape)
print(panel["country"].nunique(), "countries")
print(panel["year"].min(), "to", panel["year"].max())

(13881, 16)
29 countries
1987 to 2009


In [17]:
print(sorted(panel["country"].unique()))

['Australia', 'Austria', 'Belgium', 'Canada', 'Chile', 'Czech Republic', 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Israel', 'Italy', 'Japan', 'Korea', 'Mexico', 'Norway', 'Poland', 'Portugal', 'Slovak Republic', 'Spain', 'Sweden', 'Switzerland', 'United Kingdom', 'United States']


In [18]:
# Sort for lag construction
panel = panel.sort_values(["country", "indmatch", "year"]).reset_index(drop=True)

# Create panel identifier
panel["indctry"] = panel["country"] + "_" + panel["indmatch"]

# Construct lags
panel["l1_rdgov"]      = panel.groupby("indctry")["lnrdgov"].shift(1)
panel["l1_indctryprod"] = panel.groupby("indctry")["lnvalu"].shift(1)
panel["l1_gdp"]        = panel.groupby("indctry")["lngdp"].shift(1)
panel["l1_taxsubs"]    = panel.groupby("indctry")["taxsubs"].shift(1)

print(panel[["l1_rdgov", "l1_indctryprod", "l1_gdp", "l1_taxsubs"]].isnull().mean().round(3))

l1_rdgov          0.044
l1_indctryprod    0.044
l1_gdp            0.044
l1_taxsubs        0.152
dtype: float64


In [19]:
# Employment weights: initial share of industry employment within country
# "Initial" = first year the observation appears in the sample

# Get first available employment share for each country-industry
panel["empshare"] = panel.groupby(["country", "year"])["empn"].transform(
    lambda x: x / x.sum() * 100
)

# For each country-industry, take the employment share in the first year it appears
panel["weight"] = panel.groupby("indctry")["empshare"].transform("first")

# Check
print(panel["weight"].isnull().mean().round(3), "share missing weights")
print(panel.groupby("country")["weight"].first().head(10))

0.031 share missing weights
country
Australia          0.613801
Austria            3.212680
Belgium            2.926077
Canada             0.928602
Chile                   NaN
Czech Republic    17.032492
Denmark            3.079866
Estonia            0.133200
Finland            9.795231
France             4.616006
Name: weight, dtype: float64


## Sample restriction and saving

In [20]:
# Year trend (as in Stata: year - 1970)
panel["yeartrend"] = panel["year"] - 1970

# Numeric IDs for FEs
panel["ctrynr"]  = panel["country"].astype("category").cat.codes + 1
panel["indyear"] = (panel["indmatch"] + "_" + panel["year"].astype(str))
panel["indyear"] = panel["indyear"].astype("category").cat.codes + 1
panel["ctryyear"] = (panel["country"] + "_" + panel["year"].astype(str))
panel["ctryyear"] = panel["ctryyear"].astype("category").cat.codes + 1

# Drop rows missing core regression variables
core_vars = ["lnrdexgov", "l1_rdgov", "l1_indctryprod", "l1_gdp", "weight"]
panel_clean = panel.dropna(subset=core_vars).copy()

print(panel_clean.shape)
print(panel_clean["country"].nunique(), "countries")
print(panel_clean["indmatch"].nunique(), "industries")
print(panel_clean["year"].min(), "to", panel_clean["year"].max())

# Save to processed
panel_clean.to_csv("../data/processed/panel.csv", index=False)
print("Saved to data/processed/panel.csv")

(12897, 27)
28 countries
26 industries
1987 to 2009
Saved to data/processed/panel.csv
